In [1]:
import pandas as pd

df = pd.read_csv("customer_orders_cleaned.csv")

df.head()

,customerid,customername,gender,age,city,country,customertype,signupdate,orderid,orderdate,productid,productname,productcategory,quantity,unitprice,discount,revenue,paymentmethod,orderstatus
0,CUST01500,Aditya Bhatt,Male,36,Ahmedabad,India,New,24-07-2022,ORD516107,21-08-2024,P3148,Puzzle Set,Toys,1,2233.06,0,2233.06,Net Banking,Delivered
1,CUST00119,Simran Bose,Female,65,Bengaluru,India,New,18-05-2020,ORD503107,30-07-2023,P9055,Self-Help Book,Books,1,1349.46,5,1281.99,Credit Card,Delivered
2,CUST00746,Nikhil Mehta,Female,36,Vadodara,India,New,20-09-2023,ORD530062,17-07-2024,P4227,Smart Watch,Electronics,1,416.89,0,416.89,Credit Card,Delivered
3,CUST02385,Myra Ghosh,Female,29,Kochi,India,Premium,17-03-2023,ORD513827,19-06-2024,P6897,Lipstick,Beauty,1,2880.24,0,2880.24,Debit Card,Delivered
4,CUST01821,Rajesh Bhatt,Male,53,Kochi,India,Returning,28-10-2020,ORD526892,19-04-2023,P9367,Badminton Racket,Sports,1,836.12,5,794.31,Credit Card,Delivered


In [7]:
df["orderdate"] = pd.to_datetime(
    df["orderdate"],
    dayfirst=True,
    errors="coerce"
)

In [8]:
print(df["orderdate"].dtype)

datetime64[us]


In [9]:
analysis_date = df["orderdate"].max() + pd.Timedelta(days=1)

print("Analysis Date:", analysis_date)

Analysis Date: 2025-07-01 00:00:00


In [13]:
rfm = df.groupby("customerid").agg(
    Recency=("orderdate", lambda x: (analysis_date - x.max()).days),
    Frequency=("orderid", "nunique"),
    Monetary=("revenue", "sum")
).reset_index()

rfm.head()

,customerid,Recency,Frequency,Monetary
0,cust00102,893,1,7257.90
1,cust00113,538,1,4833.20
2,cust00129,377,1,2118.88
3,cust00147,623,1,1250.60
4,cust00179,139,1,3163.34


In [14]:
rfm.describe()

,Recency,Frequency,Monetary
count,3794.000000,3794.000000,3794.000000
mean,339.921982,2.230100,3612.779492
std,249.423869,2.714663,4942.402278
min,1.000000,1.000000,33.230000
25%,124.000000,1.000000,1069.985000
50%,294.000000,1.000000,2230.775000
75%,527.000000,2.000000,4410.367500
max,911.000000,56.000000,96538.990000


In [15]:
rfm["Frequency"].value_counts().sort_index()

Frequency
1     1958
2      890
3      421
4      198
5      127
6       54
7       30
8       29
9       19
10       9
11      17
12      10
13       6
14       5
15       2
16       3
17       1
18       2
19       1
20       1
21       2
22       1
23       1
26       1
29       1
36       1
47       1
49       1
51       1
56       1
Name: count, dtype: int64

In [16]:
rfm.head(10)

,customerid,Recency,Frequency,Monetary
0,cust00102,893,1,7257.90
1,cust00113,538,1,4833.20
2,cust00129,377,1,2118.88
3,cust00147,623,1,1250.60
4,cust00179,139,1,3163.34
5,cust00184,239,1,445.26
6,cust00200,345,1,3435.10
7,cust00205,854,1,1102.25
8,cust00237,109,2,5147.18
9,cust00250,349,1,664.19


In [17]:
rfm.describe()

,Recency,Frequency,Monetary
count,3794.000000,3794.000000,3794.000000
mean,339.921982,2.230100,3612.779492
std,249.423869,2.714663,4942.402278
min,1.000000,1.000000,33.230000
25%,124.000000,1.000000,1069.985000
50%,294.000000,1.000000,2230.775000
75%,527.000000,2.000000,4410.367500
max,911.000000,56.000000,96538.990000


In [18]:
rfm["Frequency"].value_counts().sort_index()

Frequency
1     1958
2      890
3      421
4      198
5      127
6       54
7       30
8       29
9       19
10       9
11      17
12      10
13       6
14       5
15       2
16       3
17       1
18       2
19       1
20       1
21       2
22       1
23       1
26       1
29       1
36       1
47       1
49       1
51       1
56       1
Name: count, dtype: int64

In [19]:
rfm["R_Score"] = pd.qcut(
    rfm["Recency"],
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"],
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

In [20]:
rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str) +
    rfm["F_Score"].astype(str) +
    rfm["M_Score"].astype(str)
)

rfm.head()

,customerid,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
0,cust00102,893,1,7257.90,1,1,5,115
1,cust00113,538,1,4833.20,2,1,4,214
2,cust00129,377,1,2118.88,2,1,3,213
3,cust00147,623,1,1250.60,1,1,2,112
4,cust00179,139,1,3163.34,4,1,4,414


In [21]:
rfm.describe()

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score
count,3794.000000,3794.000000,3794.000000,3794.000000,3794.000000,3794.000000
mean,339.921982,2.230100,3612.779492,3.005008,3.000000,3.000000
std,249.423869,2.714663,4942.402278,1.416905,1.414586,1.414586
min,1.000000,1.000000,33.230000,1.000000,1.000000,1.000000
25%,124.000000,1.000000,1069.985000,2.000000,2.000000,2.000000
50%,294.000000,1.000000,2230.775000,3.000000,3.000000,3.000000
75%,527.000000,2.000000,4410.367500,4.000000,4.000000,4.000000
max,911.000000,56.000000,96538.990000,5.000000,5.000000,5.000000
